# HotBob Controller Authority Probe

Use this notebook in Colab or Modal Notebooks to test the compact HotBob controller on authority behavior only. It avoids the frozen-Qwen/LoRA path and focuses on whether the controller can write, read, and act from scoped authority memories under predicted memory.


## Configure

Set `REPO_URL` if you are running against a fork. The defaults target this experiment branch.


In [ ]:
import os
from pathlib import Path

VOLUME_ROOT = os.environ.get("HOTBOB_VOLUME_ROOT", "/mnt/hotbob")
Path(VOLUME_ROOT).mkdir(parents=True, exist_ok=True)

REPO_URL = "https://github.com/Popidge/hotbob.git"
BRANCH = "experiment/authority-memory-reader"
PROJECT_DIR = f"{VOLUME_ROOT}/repo"

os.environ["UV_CACHE_DIR"] = f"{VOLUME_ROOT}/uv-cache"
Path(os.environ["UV_CACHE_DIR"]).mkdir(parents=True, exist_ok=True)

SEED = 51
TRAIN_N = 50_000
EVAL_N = 5_000
TRAIN_STEPS = 5_000
BATCH_SIZE = 32
EVAL_BATCH_SIZE = 64

TRAIN_TRACES = "data/controller_authority_train.jsonl"
EVAL_TRACES = "data/controller_authority_eval.jsonl"
CHECKPOINT = "runs/latest.pt"
REPORT = "runs/controller_authority_report.json"


## Checkout And Install

In [ ]:
import os
import subprocess
from pathlib import Path

repo_path = Path(PROJECT_DIR)
if repo_path.exists():
    os.chdir(PROJECT_DIR)
    subprocess.run(["git", "fetch", "--all", "--prune"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, PROJECT_DIR], check=True)
    os.chdir(PROJECT_DIR)

subprocess.run(["git", "checkout", BRANCH], check=True)
subprocess.run(["git", "pull", "--ff-only", "origin", BRANCH], check=True)
subprocess.run(["git", "status", "--short", "--branch"], check=True)
subprocess.run(["uv", "sync"], check=True)
print(f"Working directory: {Path.cwd()}")


## Generate Authority-Heavy Controller Data

The mix heavily weights `authority_conflict` and keeps `tool_verified_override` in the probe set so verified/unverified tool authority remains covered.


In [ ]:
!uv run python -m hotbob.data.generate \
  --train-out {TRAIN_TRACES} \
  --eval-out {EVAL_TRACES} \
  --train-n {TRAIN_N} \
  --eval-n {EVAL_N} \
  --seed {SEED} \
  --family-weight authority_conflict=10 \
  --family-weight tool_verified_override=3 \
  --family-weight stale_state_replacement=1 \
  --family-weight privacy_disclosure_conflict=1


## Train Controller

In [ ]:
!uv run python -m hotbob.training.train \
  --traces {TRAIN_TRACES} \
  --steps {TRAIN_STEPS} \
  --batch-size {BATCH_SIZE} \
  --structured-loss-weight 0.3 \
  --retrieval-contrastive-weight 0.25


## Evaluate Controller

In [ ]:
!uv run python -m hotbob.training.evaluate \
  --checkpoint {CHECKPOINT} \
  --traces {EVAL_TRACES} \
  --batch-size {EVAL_BATCH_SIZE} \
  --debug-families authority_conflict tool_verified_override \
  --debug-dumps-out runs/controller_authority_debug.jsonl


## Authority Report

In [ ]:
!uv run python -m hotbob.training.authority_report \
  --checkpoint {CHECKPOINT} \
  --traces {EVAL_TRACES} \
  --out {REPORT}


In [ ]:
import json
from pathlib import Path

report = json.loads(Path(REPORT).read_text())
for name in ["context_only", "teacher_forced", "predicted", "predicted_oracle_slot", "predicted_oracle_scope", "predicted_oracle_value", "predicted_oracle_all"]:
    row = report["modes"][name]
    print(name, {
        "authority_accuracy": row.get("authority_accuracy"),
        "authority_probe_accuracy": row.get("authority_probe_accuracy"),
        "read_mass": row.get("authority_target_slot_read_mass"),
        "boundary_f1": row.get("boundary", {}).get("f1"),
        "top_confusions": row.get("authority_top_confusions"),
    })
